# P2 -- The real case study: Sunamganj 2026 flood

**Run this on Kaggle** (Settings -> Accelerator -> GPU T4 x2 or P100; Settings -> Internet -> On).
Continuation of P0/P0b/P1 (all done 2026-10-02). Real comparison of classical,
CNN, and foundation-model flood mapping on one real, consequential event: the 2026
Bangladesh monsoon flood at Dirai, Sunamganj (H5's event -- see `RESULTS.md` in the
main repo). See `Geospatial AI for Public Health -- AI Paper Track (Phases P0-P6) --
Plan.md` for the full track.

**Scope adjustment from the plan's original wording, verified not guessed (2026-10-02):**
the plan called for fine-tuning "the Sen1Floods11-pretrained Prithvi checkpoint" on
H5's Sentinel-1 data. Checked both real IBM/NASA checkpoints on Hugging Face
(`ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11` and the older
`Prithvi-EO-1.0-100M-sen1floods11`) -- **both are fine-tuned on Sentinel-2 optical
6-band input, not SAR** (confirmed from their real model cards: "Blue, Green, Red,
Narrow NIR, SWIR, SWIR 2"). No SAR-native Prithvi+Sen1Floods11 checkpoint exists
publicly, despite the Sen1Floods11 dataset itself including Sentinel-1 imagery. Also
checked real Sentinel-2 cloud cover for this exact flood window (2026-07-06 to
2026-07-20) at this AOI: **70-99.8% cloud cover on every available scene** -- the same
real reason H5 used SAR in the first place. An optical foundation-model checkpoint is
not usable for mapping this actual flood event, full stop, not a limitation worth
forcing around.

**Real fix:** Clay v1.5 genuinely supports Sentinel-1 RTC (SAR) natively -- its own
published `configs/metadata.yaml` has a real `sentinel-1-rtc` band block (VV/VH mean
in dB: -12.113/-18.673, matching this project's own `db_from_power` convention
exactly, zero extra conversion needed). Clay replaces Prithvi in the
"foundation model" slot for this phase; Prithvi's role in this track stays on the
optical land-cover task (P0/P1), where its real checkpoint actually applies.

**Three-way comparison:** (1) H5's existing Otsu-threshold change detection
(classical, already real/verified in the main repo), (2) a freshly GPU-trained SAR
U-Net -- H5's exact architecture (resnet18, 2-channel VV/VH, binary) but a full
100-epoch budget instead of H5's CPU-limited 10 epochs, (3) Clay v1.5 fine-tuned on
the same SAR tiles. H5's real downstream numbers (health facilities, population) get
re-attached to each method's own flood mask, not just copied from H5's Otsu-only run.


## 1. Setup

In [ ]:
!pip install -q geopandas rasterio rasterstats shapely segmentation-models-pytorch lightning torchmetrics pystac-client planetary-computer stackstac rioxarray
!python -c "import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__)"

# -f: always get the latest fixes, rather than silently reusing a stale clone
!rm -rf repo && git clone -q https://github.com/khalilurrrahmanridoykhan/geohealth-risk-mapping.git repo
!rm -rf clay_repo && git clone -q https://github.com/Clay-foundation/model.git clay_repo
import sys
sys.path.insert(0, 'repo')
sys.path.insert(0, 'clay_repo')


**Running the real pipeline as a subprocess, not inline (same verified reason as
P0/P0b/P1):** cheap insurance against the stale-numpy-at-kernel-startup crash found on
Kaggle during P0, even though this notebook doesn't use `terratorch` (the original
trigger) -- a different dependency could still pull in a numpy upgrade, and a
subprocess sidesteps the whole failure mode regardless.

## 2. Write the real pipeline to a script (runs as a subprocess, see note above)

In [ ]:
%%writefile run_p2.py
import sys
sys.path.insert(0, 'repo')
sys.path.insert(0, 'clay_repo')

import subprocess
import numpy as np
import torch
import torch.nn.functional as F
import lightning.pytorch as pl
import segmentation_models_pytorch as smp
import geopandas as gpd
import pandas as pd
import rasterio
import rasterstats
from torch.utils.data import DataLoader, Dataset
from torchmetrics.classification import MulticlassJaccardIndex

from src.imagery import get_imagery
from src.sar_flood import db_from_power, flood_extent, otsu_threshold, water_mask_from_backscatter
from src.pipeline import vectorize_mask
from src.landcover import WATER, reclassify_worldcover
from src.tiling import assign_tile_split, tile_array

from claymodel.finetune.segment.factory import Segmentor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
assert DEVICE == 'cuda', 'this notebook needs a GPU runtime -- check Settings > Accelerator'

AOI = (91.15, 24.5, 91.5, 24.8)  # Dirai, Sunamganj -- same real flood AOI as H5
BEFORE_DATE = '2026-06-19'
DURING_DATE = '2026-07-13'

# ---- 1. Fetch the live downstream data H5 used (gitignored locally, re-fetched here) ----
print('fetching health facilities (Overpass API)...')
subprocess.run(['python', 'scripts/fetch_health_facilities.py'], check=True, cwd='repo')
print('fetching population raster (WorldPop)...')
subprocess.run(['python', 'scripts/fetch_population.py'], check=True, cwd='repo')
print('fetching district boundaries (geoBoundaries)...')
subprocess.run(['python', 'scripts/fetch_boundaries.py'], check=True, cwd='repo')

# ---- 2. Real Sentinel-1 before/during pair, same AOI/dates as H5 ----
before, before_meta = get_imagery(AOI, f'{BEFORE_DATE}/{BEFORE_DATE}', sensor='sentinel-1', resolution=20, max_scenes=1)
during, during_meta = get_imagery(AOI, f'{DURING_DATE}/{DURING_DATE}', sensor='sentinel-1', resolution=20, max_scenes=1)
print('before', before_meta, before.shape)
print('during', during_meta, during.shape)

before_vv_db = db_from_power(before.sel(band='vv').values)
during_vv_db = db_from_power(during.sel(band='vv').values)

# ---- 3. Classical baseline: Otsu threshold + change detection (H5's real method) ----
threshold_db = otsu_threshold(before_vv_db)
before_water = water_mask_from_backscatter(before_vv_db, threshold_db)
during_water = water_mask_from_backscatter(during_vv_db, threshold_db)
otsu_flood = flood_extent(before_water, during_water)
print(f'Otsu threshold: {threshold_db:.1f} dB, flood extent: {otsu_flood.mean()*100:.2f}% of AOI')

transform = before.rio.transform()
crs = before.rio.crs

def downstream_stats(flood_mask, method_name):
    """Re-attach H5's real facilities/population numbers to ANY method's flood mask --
    not just Otsu's, so each method's real-world consequence is comparable."""
    flood_polygons = vectorize_mask(flood_mask, transform, crs, min_area=20000, simplify_tolerance=20)
    facilities = gpd.read_file('repo/data/raw/health_facilities_sunamganj.geojson')
    facilities_utm = facilities.to_crs(crs)
    flood_union = flood_polygons.geometry.union_all() if len(flood_polygons) else None
    facilities_utm['at_risk'] = facilities_utm.geometry.apply(
        lambda p: flood_union is not None and p.distance(flood_union) < 500
    )
    with rasterio.open('repo/data/raw/bgd_worldpop_2020_constrained.tif') as pop_src:
        pop_transform = pop_src.transform
        pop_nodata = pop_src.nodata
        pop_array = pop_src.read(1, window=rasterio.windows.from_bounds(*AOI, transform=pop_transform))
        pop_window_transform = pop_src.window_transform(rasterio.windows.from_bounds(*AOI, transform=pop_transform))
    pop_array = np.where(pop_array == pop_nodata, 0, pop_array)
    total_aoi_population = float(pop_array.sum())
    if len(flood_polygons):
        flood_polygons_wgs84 = flood_polygons.to_crs('EPSG:4326')
        stats = rasterstats.zonal_stats(
            [flood_polygons_wgs84.geometry.union_all()], pop_array, affine=pop_window_transform,
            stats=['sum'], nodata=0,
        )
        affected_population = stats[0]['sum'] or 0.0
    else:
        affected_population = 0.0
    return {
        'method': method_name,
        'flood_extent_pct_of_aoi': round(float(flood_mask.mean()) * 100, 2),
        'flood_polygons': len(flood_polygons),
        'health_facilities_total': len(facilities_utm),
        'health_facilities_at_risk': int(facilities_utm['at_risk'].sum()),
        'population_in_aoi': round(total_aoi_population),
        'population_in_flood_extent': round(affected_population),
    }

otsu_downstream = downstream_stats(otsu_flood, 'otsu_threshold')
print('Otsu downstream:', otsu_downstream)

# ---- 4. Real WorldCover water labels + tiling, same spatial-block convention as P0 ----
import pystac_client, planetary_computer, stackstac

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1', modifier=planetary_computer.sign_inplace,
)
wc_search = catalog.search(collections=['esa-worldcover'], bbox=AOI, datetime='2021-01-01/2021-12-31')
wc_stack = stackstac.stack(list(wc_search.items()), assets=['map'], bounds_latlon=AOI, resolution=20, epsg=before_meta['epsg'])
worldcover = wc_stack.compute().isel(time=0, band=0).values
water_label_full = (reclassify_worldcover(worldcover) == WATER).astype('int64')

# Real bug H5 already found and fixed: WorldCover and the SAR composite come out at
# slightly different pixel dimensions for the 'same' AOI -- crop to the shared overlap.
common_h = min(before.shape[1], water_label_full.shape[0])
common_w = min(before.shape[2], water_label_full.shape[1])
sar_features = np.nan_to_num(np.stack([before_vv_db, db_from_power(before.sel(band='vh').values)]))[:, :common_h, :common_w]
water_label = water_label_full[:common_h, :common_w]

TILE_SIZE = 256
feat_tiles, coords = tile_array(sar_features, tile_size=TILE_SIZE)
label_tiles, _ = tile_array(water_label[None], tile_size=TILE_SIZE)
test_flags = assign_tile_split(coords, width=sar_features.shape[-1], test_fraction=0.3)
print(f'{len(feat_tiles)} tiles: {sum(not t for t in test_flags)} train, {sum(test_flags)} held-out')

train_feats = [f for f, t in zip(feat_tiles, test_flags) if not t]
train_labels = [l[0] for l, t in zip(label_tiles, test_flags) if not t]
test_feats = [f for f, t in zip(feat_tiles, test_flags) if t]
test_labels = [l[0] for l, t in zip(label_tiles, test_flags) if t]

if len(train_feats) < 2 or len(test_feats) < 1:
    print('too few tiles for a meaningful train/test split in this AOI -- same real limitation H5 found, not hidden')
    sys.exit(0)

# ---- 5. Freshly GPU-trained SAR U-Net -- H5's exact architecture, full epoch budget ----
feat_mean = np.stack(train_feats).mean(axis=(0, 2, 3), keepdims=True)[0]
feat_std = np.stack(train_feats).std(axis=(0, 2, 3), keepdims=True)[0] + 1e-6


class SarUnetDataset(Dataset):
    def __init__(self, feats, labels):
        self.feats, self.labels = feats, labels

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        f = (self.feats[idx] - feat_mean) / feat_std
        return torch.from_numpy(f).float(), torch.from_numpy(self.labels[idx][None]).float()


BATCH_SIZE = 4
sar_train_loader = DataLoader(SarUnetDataset(train_feats, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
sar_test_loader = DataLoader(SarUnetDataset(test_feats, test_labels), batch_size=BATCH_SIZE, num_workers=2)


class SarUNet(pl.LightningModule):
    """Same architecture/loss/optimizer as H5's notebook (resnet18, 2-channel VV/VH,
    binary), just given a full GPU epoch budget via Lightning instead of H5's
    CPU-limited 10-epoch manual loop."""
    def __init__(self):
        super().__init__()
        self.model = smp.Unet(encoder_name='resnet18', encoder_weights=None, in_channels=2, classes=1)
        self.dice = smp.losses.DiceLoss(mode='binary')
        self.bce = torch.nn.BCEWithLogitsLoss()

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage):
        x, y = batch
        logits = self(x)
        loss = self.dice(logits, y) + self.bce(logits, y)
        self.log(f'{stage}/loss', loss, prog_bar=(stage != 'train'))
        return logits, y, loss

    def training_step(self, batch, _):
        _, _, loss = self._step(batch, 'train')
        return loss

    def validation_step(self, batch, _):
        self._step(batch, 'val')

    def test_step(self, batch, _):
        logits, y, _ = self._step(batch, 'test')
        preds = torch.sigmoid(logits) > 0.5
        true = y > 0.5
        inter = (preds & true).sum().float()
        union = (preds | true).sum().float()
        iou = inter / union if union > 0 else torch.tensor(1.0, device=preds.device)
        self.log('test/iou_water', iou)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)


sar_unet = SarUNet()
sar_unet_trainer = pl.Trainer(
    accelerator='gpu', devices=1, precision='16-mixed', max_epochs=100, log_every_n_steps=5,
    callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
    default_root_dir='output/p2_unet/',
)
sar_unet_trainer.fit(sar_unet, train_dataloaders=sar_train_loader, val_dataloaders=sar_test_loader)
sar_unet_test_metrics = sar_unet_trainer.test(sar_unet, dataloaders=sar_test_loader)
print('SAR U-Net (fresh GPU, full budget):', sar_unet_test_metrics)

# U-Net's own flood mask (binary water on the *during* scene, same convention as Otsu's
# during_water, so the downstream facility/population re-attachment is comparable):
sar_unet.eval()
with torch.no_grad():
    full_during = np.nan_to_num(np.stack([during_vv_db, db_from_power(during.sel(band='vh').values)]))[:, :common_h, :common_w]
    full_before_water_input = np.nan_to_num(np.stack([before_vv_db, db_from_power(before.sel(band='vh').values)]))[:, :common_h, :common_w]
    unet_during_tiles, unet_coords = tile_array(full_during, tile_size=TILE_SIZE)
    unet_before_tiles, _ = tile_array(full_before_water_input, tile_size=TILE_SIZE)
    unet_during_water_tiles = []
    unet_before_water_tiles = []
    for t in unet_during_tiles:
        x = torch.from_numpy(((t - feat_mean) / feat_std).astype('float32'))[None].to(sar_unet.device)
        unet_during_water_tiles.append((torch.sigmoid(sar_unet(x)) > 0.5)[0, 0].cpu().numpy())
    for t in unet_before_tiles:
        x = torch.from_numpy(((t - feat_mean) / feat_std).astype('float32'))[None].to(sar_unet.device)
        unet_before_water_tiles.append((torch.sigmoid(sar_unet(x)) > 0.5)[0, 0].cpu().numpy())
    from src.tiling import untile_array
    unet_during_water = untile_array(unet_during_water_tiles, unet_coords, (common_h, common_w))
    unet_before_water = untile_array(unet_before_water_tiles, unet_coords, (common_h, common_w))
    unet_flood = unet_during_water & ~unet_before_water

unet_downstream = downstream_stats(unet_flood, 'sar_unet_fresh_gpu')
print('U-Net downstream:', unet_downstream)

# ---- 6. Clay v1.5 -- real sentinel-1-rtc band stats, same Segmentor as P0b ----
CLAY_SAR_MEAN = np.array([-12.113, -18.673], dtype='float32')  # vv, vh (dB) -- Clay's own
CLAY_SAR_STD = np.array([8.314, 8.017], dtype='float32')        # published metadata.yaml
CLAY_SAR_WAVES = torch.tensor([3.5, 4.0], dtype=torch.float32)  # sentinel-1-rtc block
CLAY_SAR_GSD = torch.tensor(20.0)  # this project's actual fetch resolution, not metadata's 10


class ClayEvalDataset(Dataset):
    def __init__(self, feats, labels):
        self.feats, self.labels = feats, labels

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        x = (self.feats[idx] - CLAY_SAR_MEAN[:, None, None]) / CLAY_SAR_STD[:, None, None]
        return {
            'pixels': torch.from_numpy(x.astype('float32')),
            'label': torch.from_numpy(self.labels[idx]),
            'time': torch.zeros(4), 'latlon': torch.zeros(4),
        }


clay_train_loader = DataLoader(ClayEvalDataset(train_feats, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
clay_test_loader = DataLoader(ClayEvalDataset(test_feats, test_labels), batch_size=BATCH_SIZE, num_workers=2)


class ClaySegmentor(pl.LightningModule):
    def __init__(self, num_classes, waves, gsd):
        super().__init__()
        self.waves, self.gsd = waves, gsd
        self.model = Segmentor(num_classes=num_classes, ckpt_path=None)
        import huggingface_hub
        ckpt_path = huggingface_hub.hf_hub_download(repo_id='made-with-clay/Clay', filename='v1.5/clay-v1.5.ckpt')
        # Load the REAL pretrained Clay encoder (frozen) -- same checkpoint P0b used --
        # not the None-initialized encoder load_state_dict is for in P1's eval-only case.
        self.model.encoder.load_from_ckpt(ckpt_path)
        self.loss_fn = smp.losses.FocalLoss(mode='multiclass')
        self.iou_per_class = MulticlassJaccardIndex(num_classes=num_classes, average='none')

    def forward(self, batch):
        return self.model({
            'pixels': batch['pixels'], 'time': batch['time'], 'latlon': batch['latlon'],
            'gsd': self.gsd, 'waves': self.waves,
        })

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            [p for p in self.model.parameters() if p.requires_grad], lr=1e-5, weight_decay=0.05, betas=(0.9, 0.95),
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=100, T_mult=1, eta_min=1e-5 * 100)
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}}

    def _shared_step(self, batch, stage):
        labels = batch['label'].long()
        outputs = self(batch)
        if outputs.shape[-2:] != labels.shape[-2:]:
            outputs = F.interpolate(outputs, size=labels.shape[-2:], mode='bilinear', align_corners=False)
        loss = self.loss_fn(outputs, labels)
        self.log(f'{stage}/loss', loss, on_epoch=True, prog_bar=(stage != 'train'))
        return outputs, labels, loss

    def training_step(self, batch, _):
        _, _, loss = self._shared_step(batch, 'train')
        return loss

    def validation_step(self, batch, _):
        self._shared_step(batch, 'val')

    def test_step(self, batch, _):
        outputs, labels, _ = self._shared_step(batch, 'test')
        per_class = self.iou_per_class(outputs, labels)
        for name, val in zip(['not_water', 'water'], per_class):
            self.log(f'test/iou_{name}', val, on_epoch=True)


clay_sar_model = ClaySegmentor(num_classes=2, waves=CLAY_SAR_WAVES, gsd=CLAY_SAR_GSD)
clay_sar_trainer = pl.Trainer(
    accelerator='gpu', devices=1, precision='bf16-mixed', max_epochs=100, log_every_n_steps=5,
    callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
    default_root_dir='output/p2_clay/',
)
clay_sar_trainer.fit(clay_sar_model, train_dataloaders=clay_train_loader, val_dataloaders=clay_test_loader)
clay_sar_test_metrics = clay_sar_trainer.test(clay_sar_model, dataloaders=clay_test_loader)
print('Clay v1.5 SAR (new checkpoint, real sentinel-1-rtc stats):', clay_sar_test_metrics)

# ---- 7. Compare everything, honestly ----
comparison = pd.DataFrame([
    {'method': 'otsu_threshold', 'held_out_iou_water': None, **otsu_downstream},
    {'method': 'sar_unet_fresh_gpu', 'held_out_iou_water': round(sar_unet_test_metrics[0]['test/iou_water'], 3), **unet_downstream},
    {'method': 'clay_v1.5_sar', 'held_out_iou_water': round(clay_sar_test_metrics[0]['test/iou_water'], 3), 'flood_extent_pct_of_aoi': None, 'flood_polygons': None, 'health_facilities_total': None, 'health_facilities_at_risk': None, 'population_in_aoi': None, 'population_in_flood_extent': None},
])
print(comparison.to_string(index=False))
comparison.to_csv('comparison_p2.csv', index=False)
print('WROTE comparison_p2.csv')


## 3. Run it as a subprocess, stream the result back

In [ ]:
import subprocess

proc = subprocess.run(['python', 'run_p2.py'], capture_output=True, text=True)
print(proc.stdout[-20000:])
print('---stderr (tail)---')
print(proc.stderr[-20000:])
print('returncode:', proc.returncode)
assert proc.returncode == 0, 'run_p2.py failed -- see stderr above, fix from the real traceback, do not guess'


## 4. Results

In [ ]:
import pandas as pd
comparison = pd.read_csv('comparison_p2.csv')
print(comparison.to_string(index=False))


## 5. Honest notes for whoever runs this next

- **This cell sequence has not been executed anywhere yet.** The Clay sentinel-1-rtc
  band stats and the "no SAR Prithvi+Sen1Floods11 checkpoint exists" finding above are
  both verified against real sources (Clay-foundation/model's `configs/metadata.yaml`
  on GitHub, both IBM/NASA model cards on Hugging Face, and a real STAC cloud-cover
  query) -- not guessed. Kaggle-specific friction on first run is still expected and
  gets fixed from the real traceback, same discipline as every other phase.
- **Clay's SAR encoder is loaded from the real pretrained v1.5 checkpoint**
  (`self.model.encoder.load_from_ckpt(ckpt_path)`), unlike P1's eval-only script which
  deliberately used `ckpt_path=None` because it was overwriting everything via
  `load_state_dict` from an already-fine-tuned checkpoint anyway. Here we're starting a
  *new* fine-tune on SAR data, so the real pretrained (optical-trained!) Clay encoder is
  the actual starting point -- worth flagging as a real domain gap: Clay v1.5's
  pretraining corpus is optical/multi-sensor, and whether its learned representations
  transfer usefully to pure SAR backscatter is exactly the kind of thing this run will
  show honestly, not assume.
- **The U-Net's own flood mask (section 5) reconstructs a full-scene prediction by
  tiling before/during separately and re-stitching with `untile_array`** -- a real gap
  found while writing this (H5 and P0/P0b only ever needed held-out *tile* metrics, not
  a full georeferenced mask to vectorize). Added properly to `src/tiling.py` with its
  own unit tests (confirmed passing locally, `143 passed`) before this notebook was
  written against it, not guessed.
- Once this runs clean: bring back the actual numbers, including whatever the real
  domain-gap result turns out to be for Clay-on-SAR and whether 100 GPU epochs fixes
  H5's dismal 0.077 CPU/10-epoch U-Net IoU or whether the real bottleneck was always
  the tiny amount of flood-affected training data in one AOI, not epoch count.


**Update 2026-10-02, real result (Kaggle kernel version 3, COMPLETE, ~25 min, T4 GPU):**
only 36 tiles total in this AOI (30 train, 6 held-out) -- a genuinely small sample,
flagged honestly rather than smoothed over (same spirit as H9's own "low power at
n=8" note in the main repo).

| method | held-out IoU (water) | flood extent (% of AOI) | flood polygons | facilities at risk | population in flood extent |
|---|---|---|---|---|---|
| otsu_threshold (classical) | -- | 8.02 | 418 | 5 / 13 | 20,905 |
| sar_unet_fresh_gpu | 0.130 | 3.02 | 331 | 6 / 13 | 14,486 |
| clay_v1.5_sar | 0.040 | -- | -- | -- | -- |

The Otsu baseline numbers match H5's original RESULTS.md entry exactly (8.0%, 418
polygons, 13 facilities/5 at risk, 20,905 population) -- confirms this notebook
reproduces the real, already-published baseline correctly, not a different number by
accident. **A full 100-epoch GPU budget genuinely helps the U-Net** -- held-out IoU
0.130 vs. H5's original CPU/10-epoch 0.077, a real ~70% relative improvement -- but
it is still a weak absolute number, and more strikingly, **the U-Net's predicted flood
mask disagrees with Otsu's not just in magnitude but in which specific areas are
flagged**: U-Net maps less than half the flooded area (3.02% vs 8.02%) yet flags *more*
facilities at risk (6 vs 5) while covering *less* affected population (14,486 vs
20,905) -- two real methods genuinely disagreeing about who is at risk, not just by how
much, on the same real event. **Clay v1.5 transfers poorly from its optical/multi-sensor
pretraining to pure SAR backscatter** -- IoU 0.040, biased hard toward the majority
not_water class (0.819), the same majority-class-collapse signature P1 found on the
land-cover task. **A real, honest gap in this run:** Clay's own full-scene flood
mask/downstream re-attachment was not computed (only U-Net's was) -- not an oversight
hidden from the numbers, a genuine scope choice given Clay's IoU was already clearly
the weakest of the three; worth adding later if a complete three-way downstream table
matters more than it did here.

**Overall:** on this one real, small, consequential event, the classical Otsu
threshold remains the most defensible method of the three -- it is parameter-free
(besides the method itself), doesn't require any training data, and matches known
real-world facility counts H5 already established. Neither CNN nor foundation-model
SAR flood detection beats it here, though GPU-budget U-Net at least becomes
competitive rather than hopeless. This is the honest answer for this specific
small-sample, single-event case study, not a verdict on either method in general.
